# train.ipynb — search, final training, calibration, operating points (ViT-Tiny / CIFAR-10, Kaggle T4)

**Flow:** profile a *measured-energy* cost model -> AutoML search over (exit placement, head architecture, token-merge ratio),
scored over a whole τ (entropy-threshold) sweep -> pick candidates from the energy-driven front **and** from a MAC-driven front
(the NACHOS-style cost, as the ablation) -> train the final models -> calibrate -> choose conservative / knee / aggressive operating
points on a held-out validation split -> save everything to `/kaggle/working`.

**What changed vs. the first version (all from reading your results):**
- Exit heads now start with a LayerNorm; the optimiser has a one-cycle schedule; data lives on the GPU (no DataLoader workers) with fp16 autocast.
- Calibration is a robust grid search over T (the LBFGS version collapsed to its floor on 3 heads).
- τ is no longer hard-coded: every (config, τ) is a point, so energy-vs-accuracy trade-offs and the knee are real.
- Energy comes from the NVML counter of GPU 0 (no CodeCarbon, no second idle GPU, no sub-second windows). During search, energy is
  *predicted* from the measured per-block cost model; `evaluate.ipynb` then verifies it with real single-sample runs.
- Two pipelines on the same archive: **energy-driven (ours)** vs **MAC-driven (NACHOS-style cost)**, evaluated later on *measured* energy.
- Token-merge ratio now means what it says (fraction of patch tokens removed); before, the code removed only half of that.
- Resumable: search state is saved after every generation, final models are skipped if their checkpoint exists.

**Audit fixes (2026-11-01):**
- Input normalisation now matches the weights (0.5/0.5, asserted against timm's config). The ImageNet stats used before held B0 at about 79% instead of about 93%, so **all earlier results are invalid**.
- The B0 sanity gate now stops the run instead of printing a warning.
- The dataset cache moved out of `/kaggle/working`, so outputs are about 1 MB rather than 170 MB.
- Runs on Kaggle, Colab or locally; CIFAR-10 is read from the official archive or downloaded with a checksum.
- Energy windows are longer: 10 s per micro-benchmark, 2,000 samples and 5 repeats for real energy.

**Stated simplifications (put in the paper):** frozen ImageNet-pretrained backbone (linear-probe-style training); one shared τ across exits;
token merging is a simplified single-point ToMe; the search is a small NSGA-II-style loop, not a supernet NAS.

## 0. Install & imports

In [ ]:
!pip install -q timm nvidia-ml-py


In [ ]:
# ================= CONFIG (identical block in train.ipynb and evaluate.ipynb) =================
import os, json, time, copy, random, hashlib, math, zipfile
from contextlib import nullcontext
import numpy as np

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"   # only used for my offline test, leave as is on Kaggle

ON_KAGGLE = os.path.isdir("/kaggle/working")
# Data lookup order: cached tensors -> cifar-10-binary.tar.gz in DATA_DIR -> train/ + test/ folders in DATA_DIR -> download.
DATA_DIR = os.environ.get("CIFAR_DIR", "/kaggle/input/datasets/ayush1220/cifar10/cifar10" if ON_KAGGLE else "data/cifar-10")
WORKING_DIR = "/tmp/smoke_working" if SMOKE_TEST else ("/kaggle/working" if ON_KAGGLE else "outputs")
CHECKPOINT_DIR = os.path.join(WORKING_DIR, "checkpoints")
RESULTS_DIR = os.path.join(WORKING_DIR, "results")
# Never inside WORKING_DIR: on Kaggle a cache there is saved as output and bloated the download from ~1 MB to 170 MB.
CACHE_DIR = "/tmp/cifar_cache"
for _d in [CHECKPOINT_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(_d, exist_ok=True)

BACKBONE_NAME = "vit_tiny_patch16_224"
IMG_SIZE = 64 if SMOKE_TEST else 224
NUM_CLASSES = 10
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
SEED = 42
USE_AMP = not SMOKE_TEST          # fp16 autocast on GPU. If you ever hit a dtype error, set False.

# ---- search space ----
EXIT_LAYER_POOL = [2, 4, 6, 8, 10]            # ViT-Tiny has 12 blocks (12 = final classifier)
HEAD_ARCH_POOL = [{"arch": "linear"}, {"arch": "mlp", "hidden": 128}, {"arch": "mlp", "hidden": 192}]
MERGE_RATIO_POOL = [0.0, 0.25, 0.5]           # fraction of patch tokens removed at that exit (max ~0.5 by construction)
MAX_EXITS = 2
TAU_GRID = [0.0, 0.02, 0.05, 0.08, 0.12, 0.16, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  # shared entropy threshold

# ---- data budget (all slices are disjoint, carved from the 50k train images) ----
SEARCH_TRAIN_PER_CLASS = 600      # 6,000
SEARCH_VAL_PER_CLASS = 150        # 1,500
SEARCH_CAL_PER_CLASS = 50         # 500
SEARCH_EPOCHS = 4
SEARCH_POP = 12
SEARCH_GENS = 5
FINAL_TRAIN_PER_CLASS = 2000      # 20,000  (raise towards 4,500 for more data; test set is separate)
FINAL_CAL_PER_CLASS = 100         # 1,000   (temperature scaling)
FINAL_VAL_PER_CLASS = 200         # 2,000   (threshold / operating-point selection)
FINAL_EPOCHS = 10

EPS = 0.01                        # no-regression tolerance: accuracy >= baseline - EPS
LR = 2e-3
BATCH_SIZE = 128

# ---- measurement ----
N_ENERGY_REPEATS = 5
REAL_ENERGY_SAMPLES = 2000        # test images for the real single-sample early-exit energy pass
PROFILE_TARGET_SECONDS = 10.0     # how long each cost-model micro-benchmark runs
CARBON_INTENSITY_KG_PER_KWH = 0.4 # ASSUMPTION (grid average). Change to your region's figure; state it in the paper.

if SMOKE_TEST:
    SEARCH_TRAIN_PER_CLASS, SEARCH_VAL_PER_CLASS, SEARCH_CAL_PER_CLASS = 8, 4, 3
    SEARCH_EPOCHS, SEARCH_POP, SEARCH_GENS = 1, 3, 2
    FINAL_TRAIN_PER_CLASS, FINAL_CAL_PER_CLASS, FINAL_VAL_PER_CLASS = 20, 5, 6
    FINAL_EPOCHS = 2
    REAL_ENERGY_SAMPLES, N_ENERGY_REPEATS, PROFILE_TARGET_SECONDS = 20, 2, 0.05


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import pandas as pd

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

device = "cuda" if torch.cuda.is_available() else "cpu"
# Must match the pretrained weights. augreg ViT-Tiny expects 0.5/0.5; ImageNet stats here cost about 14 accuracy points.
MODEL_MEAN = torch.tensor((0.5, 0.5, 0.5), device=device).view(1, 3, 1, 1)
MODEL_STD = torch.tensor((0.5, 0.5, 0.5), device=device).view(1, 3, 1, 1)

def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if (USE_AMP and device == "cuda") else nullcontext()

set_seed(SEED)
print("device:", device, "| AMP:", USE_AMP and device == "cuda", "| smoke test:", SMOKE_TEST)


## 1. Data (decode once -> GPU)

In [ ]:
# ---------------- data: decode once, keep uint8 on the GPU, resize/normalize on the GPU ----------------
# (No DataLoader workers -> removes the CPU decode/resize bottleneck and the worker hangs on Kaggle.)
def find_split_dir(root, split_name):
    for dirpath, dirnames, _ in os.walk(root):
        if os.path.basename(dirpath) == split_name:
            return dirpath
    raise FileNotFoundError(f"No '{split_name}' directory under {root}")

def read_split_to_uint8(split_dir):
    from PIL import Image
    from torchvision import datasets
    ds = datasets.ImageFolder(split_dir)
    n = len(ds)
    X = np.empty((n, 32, 32, 3), dtype=np.uint8)
    Y = np.empty((n,), dtype=np.int64)
    for i, (path, label) in enumerate(ds.samples):
        img = Image.open(path).convert("RGB")
        if img.size != (32, 32):
            img = img.resize((32, 32), Image.BILINEAR)
        X[i] = np.asarray(img); Y[i] = label
        if (i + 1) % 10000 == 0:
            print(f"  decoded {i + 1}/{n}")
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(Y), ds.classes

CIFAR_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz"
CIFAR_MD5 = "c32a1d4ab5d03f1284b67883e8d87530"

def read_cifar_tar(path):
    # Official binary format: each record is 1 label byte + 3072 image bytes. Raw bytes, so no pickle is involved.
    import tarfile
    def part(tf, names):
        raw = b"".join(tf.extractfile(m).read() for m in sorted(names))
        a = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3073)
        return torch.from_numpy(a[:, 1:].reshape(-1, 3, 32, 32).copy()), torch.from_numpy(a[:, 0].astype(np.int64))
    with tarfile.open(path) as tf:
        names = tf.getnames()
        Xtr, Ytr = part(tf, [n for n in names if "data_batch_" in n])
        Xte, Yte = part(tf, [n for n in names if "test_batch" in n])
    assert len(Ytr) == 50000 and len(Yte) == 10000, "unexpected CIFAR-10 archive contents"
    return Xtr, Ytr, Xte, Yte

def download_cifar():
    import urllib.request
    path = os.path.join(CACHE_DIR, "cifar-10-binary.tar.gz")
    print("downloading CIFAR-10 (162 MB):", CIFAR_URL)
    urllib.request.urlretrieve(CIFAR_URL, path)
    md5 = hashlib.md5(open(path, "rb").read()).hexdigest()
    assert md5 == CIFAR_MD5, f"CIFAR-10 checksum mismatch: {md5}"
    return path

def load_cifar_tensors():
    if SMOKE_TEST:
        g = torch.Generator().manual_seed(0)
        def synth(n):
            Y = torch.arange(n) % NUM_CLASSES
            X = (torch.rand(n, 3, 32, 32, generator=g) * 80 + Y.view(-1, 1, 1, 1) * 15).clamp(0, 255).to(torch.uint8)
            return X, Y.long()
        Xtr, Ytr = synth(800); Xte, Yte = synth(300)
        return Xtr, Ytr, Xte, Yte
    cache = os.path.join(CACHE_DIR, "cifar10_uint8.pt")
    if os.path.exists(cache):
        d = torch.load(cache)
        return d["Xtr"], d["Ytr"], d["Xte"], d["Yte"]
    tar = os.path.join(DATA_DIR, "cifar-10-binary.tar.gz")
    if os.path.isfile(tar):
        Xtr, Ytr, Xte, Yte = read_cifar_tar(tar)
    elif os.path.isdir(DATA_DIR):
        train_dir, test_dir = find_split_dir(DATA_DIR, "train"), find_split_dir(DATA_DIR, "test")
        print("train dir:", train_dir, "| test dir:", test_dir)
        Xtr, Ytr, classes = read_split_to_uint8(train_dir)
        Xte, Yte, _ = read_split_to_uint8(test_dir)
        assert list(classes) == CLASS_NAMES, f"class order differs from CLASS_NAMES: {classes}"
    else:
        Xtr, Ytr, Xte, Yte = read_cifar_tar(download_cifar())
    torch.save({"Xtr": Xtr, "Ytr": Ytr, "Xte": Xte, "Yte": Yte}, cache)
    return Xtr, Ytr, Xte, Yte

def prepare_batch(xb_uint8, augment=False):
    x = xb_uint8.float().div_(255.0)
    if augment:
        flip = torch.rand(x.size(0), device=x.device) < 0.5
        x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    x = F.interpolate(x, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    return (x - MODEL_MEAN) / MODEL_STD

class GPUData:
    def __init__(self, X, Y):
        self.X, self.Y = X.to(device), Y.to(device)
    def batches(self, idx, batch_size, shuffle=False, augment=False):
        idx_t = torch.as_tensor(idx, dtype=torch.long, device=device)
        if shuffle:
            idx_t = idx_t[torch.randperm(len(idx_t), device=device)]
        for i in range(0, len(idx_t), batch_size):
            b = idx_t[i:i + batch_size]
            yield prepare_batch(self.X[b], augment), self.Y[b]

def stratified_indices(labels_np, per_class, seed, exclude=None):
    exclude = exclude or set()
    rng = np.random.RandomState(seed)
    chosen = []
    for c in range(NUM_CLASSES):
        idxs = np.array([i for i in np.where(labels_np == c)[0] if i not in exclude])
        rng.shuffle(idxs)
        chosen.extend(idxs[:per_class].tolist())
    return chosen


## 2. Model, cost models, calibration/sweep/front utilities

In [ ]:
# ---------------- model: frozen ViT backbone + exit heads + token merging ----------------
def merge_pair_count(Np, ratio):
    # number of patch tokens removed. ratio = fraction of patch tokens removed (capped: bipartite matching can
    # remove at most one token per token of set A, so the practical maximum is ~0.5).
    if ratio <= 0 or Np < 4:
        return 0
    a_len = (Np + 1) // 2
    return int(min(max(1, int(Np * ratio)), a_len - 1))

def merged_patch_count(Np, ratio):
    return Np - merge_pair_count(Np, ratio)

def bipartite_soft_matching_merge(tokens, ratio):
    # Simplified ToMe-style merge: split patch tokens into sets A/B, match every A token to its most similar B token
    # (cosine), merge the r most similar pairs by mean. CLS (index 0) is protected. Done in fp32 for safety.
    B, N, D = tokens.shape
    r = merge_pair_count(N - 1, ratio)
    if r <= 0:
        return tokens
    dtype = tokens.dtype
    t = tokens.float()
    cls_tok, patch = t[:, :1], t[:, 1:]
    a, b = patch[:, 0::2], patch[:, 1::2]
    with torch.no_grad():
        scores = F.normalize(a, dim=-1) @ F.normalize(b, dim=-1).transpose(-1, -2)
        node_max, node_idx = scores.max(dim=-1)
        order = node_max.argsort(dim=-1, descending=True)
        src_idx, unm_idx = order[:, :r], order[:, r:]
        dst_idx = node_idx.gather(-1, src_idx)
    D_ = a.shape[-1]
    unm = a.gather(1, unm_idx.unsqueeze(-1).expand(-1, -1, D_))
    src = a.gather(1, src_idx.unsqueeze(-1).expand(-1, -1, D_))
    b = b.scatter_reduce(1, dst_idx.unsqueeze(-1).expand(-1, -1, D_), src, reduce="mean", include_self=True)
    return torch.cat([cls_tok, unm, b], dim=1).to(dtype)

def embed_tokens(backbone, x):
    x = backbone.patch_embed(x)
    x = backbone._pos_embed(x)
    if hasattr(backbone, "norm_pre"):
        x = backbone.norm_pre(x)
    return x

class OffRampHead(nn.Module):
    def __init__(self, dim, num_classes, arch="linear", hidden=128):
        super().__init__()
        body = nn.Linear(dim, num_classes) if arch == "linear" else \
            nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, num_classes))
        # LayerNorm first: intermediate residual-stream CLS tokens are un-normalised and their scale grows with depth.
        self.net = nn.Sequential(nn.LayerNorm(dim), body)
    def forward(self, x):
        return self.net(x)

def norm_cfg(cfg):
    return {"exit_layers": sorted(int(l) for l in cfg["exit_layers"]),
            "head_arch": {int(k): dict(v) for k, v in cfg["head_arch"].items()},
            "merge_ratio": {int(k): float(v) for k, v in cfg["merge_ratio"].items()}}

def cfg_hash(cfg):
    return hashlib.md5(json.dumps(norm_cfg(cfg), sort_keys=True).encode()).hexdigest()[:8]

def cfg_label(cfg):
    cfg = norm_cfg(cfg)
    if not cfg["exit_layers"]:
        return "full"
    parts = []
    for l in cfg["exit_layers"]:
        h = cfg["head_arch"][l]
        hs = "lin" if h["arch"] == "linear" else f"mlp{h['hidden']}"
        parts.append(f"L{l}:{hs}:m{cfg['merge_ratio'].get(l, 0.0)}")
    return " + ".join(parts)

class EarlyExitViT(nn.Module):
    def __init__(self, backbone, num_classes, exit_layers=None, head_arch=None, merge_ratio=None):
        super().__init__()
        self.backbone = backbone
        self.exit_layers = sorted(exit_layers or [])
        head_arch, merge_ratio = head_arch or {}, merge_ratio or {}
        self.heads = nn.ModuleDict({
            str(l): OffRampHead(backbone.embed_dim, num_classes, **head_arch.get(l, {"arch": "linear"}))
            for l in self.exit_layers})
        self.merge_ratio = {l: merge_ratio.get(l, 0.0) for l in self.exit_layers}
        self.final_head = nn.Linear(backbone.embed_dim, num_classes)
        self.temps = {l: 1.0 for l in self.exit_layers}; self.temps["final"] = 1.0

    def forward(self, x):
        # Batched: every sample runs the full network and we return ALL exit logits. Exit decisions are applied
        # offline (see sweep_points). The frozen backbone runs under no_grad; only heads receive gradients.
        with torch.no_grad():
            tokens = embed_tokens(self.backbone, x)
        all_logits = {}
        for i, block in enumerate(self.backbone.blocks, start=1):
            with torch.no_grad():
                tokens = block(tokens)
            if i in self.exit_layers:
                all_logits[i] = self.heads[str(i)](tokens[:, 0, :]).float()
                if self.merge_ratio.get(i, 0.0) > 0:
                    with torch.no_grad():
                        tokens = bipartite_soft_matching_merge(tokens, self.merge_ratio[i])
        with torch.no_grad():
            feat = self.backbone.norm(tokens)[:, 0, :]
        all_logits["final"] = self.final_head(feat).float()
        return all_logits

    @torch.no_grad()
    def forward_single_early_exit(self, x, tau):
        # x: [1, C, H, W]. TRUE early exit: the loop really stops, so the remaining blocks are never executed.
        tokens = embed_tokens(self.backbone, x)
        for i, block in enumerate(self.backbone.blocks, start=1):
            tokens = block(tokens)
            if i in self.exit_layers:
                logits = self.heads[str(i)](tokens[:, 0, :]).float()
                p = F.softmax(logits / self.temps.get(i, 1.0), dim=-1)
                ent = -(p * torch.log(p.clamp_min(1e-12))).sum() / math.log(p.numel())
                if ent.item() < tau:
                    return logits, i
                if self.merge_ratio.get(i, 0.0) > 0:
                    tokens = bipartite_soft_matching_merge(tokens, self.merge_ratio[i])
        feat = self.backbone.norm(tokens)[:, 0, :]
        return self.final_head(feat).float(), "final"

def save_ckpt(model, cfg, temps, path):
    torch.save({"config": norm_cfg(cfg), "final_head": model.final_head.state_dict(),
                "heads": {str(l): model.heads[str(l)].state_dict() for l in model.exit_layers},
                "temps": {str(k): float(v) for k, v in temps.items()}}, path)

def parse_temps(d):
    return {(int(k) if str(k).isdigit() else k): float(v) for k, v in d.items()}

def load_ckpt(path, backbone):
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = norm_cfg(ck["config"])
    model = EarlyExitViT(backbone, NUM_CLASSES, **cfg).to(device)
    model.final_head.load_state_dict(ck["final_head"])
    for l, sd in ck["heads"].items():
        model.heads[str(l)].load_state_dict(sd)
    model.temps = parse_temps(ck["temps"])
    model.eval()
    return model, cfg, model.temps

@torch.no_grad()
def collect_logits(model, data, idx, batch_size=256):
    model.eval()
    outs, labels = {}, []
    for x, y in data.batches(idx, batch_size):
        with amp_ctx():
            lg = model(x)
        for k, v in lg.items():
            outs.setdefault(k, []).append(v.float().cpu())
        labels.append(y.cpu())
    return {k: torch.cat(v) for k, v in outs.items()}, torch.cat(labels)

def make_backbone():
    kw = {"img_size": IMG_SIZE} if SMOKE_TEST else {}
    bb = timm.create_model(BACKBONE_NAME, pretrained=not SMOKE_TEST, num_classes=0, **kw).to(device)
    cfg_mean, cfg_std = bb.pretrained_cfg["mean"], bb.pretrained_cfg["std"]
    assert np.allclose(cfg_mean, MODEL_MEAN.flatten().tolist()) and np.allclose(cfg_std, MODEL_STD.flatten().tolist()), \
        f"input normalisation does not match the weights: expected mean={cfg_mean} std={cfg_std}"
    for p in bb.parameters():
        p.requires_grad = False
    bb.eval()
    geom = dict(num_patches=bb.patch_embed.num_patches, dim=bb.embed_dim, blocks=len(bb.blocks))
    return bb, geom


In [ ]:
# ---------------- cost models: measured energy (NVML) vs analytic MACs vs token-layers ----------------
class EnergyMeter:
    # Reads the cumulative GPU energy counter via NVML (millijoules) on GPU 0 only -> real measured joules.
    # Falls back to wall-clock seconds if NVML is unavailable (unit then reads "s"; fine for relative comparisons).
    def __init__(self, gpu_index=0):
        self.ok, self.unit = False, "s"
        if device == "cuda":
            try:
                import pynvml
                pynvml.nvmlInit()
                self.nv = pynvml
                self.h = pynvml.nvmlDeviceGetHandleByIndex(gpu_index)
                pynvml.nvmlDeviceGetTotalEnergyConsumption(self.h)
                self.ok, self.unit = True, "J"
            except Exception as e:
                print("NVML energy counter unavailable, falling back to wall-clock seconds:", e)
        print("EnergyMeter unit:", self.unit)
    def read(self):
        if device == "cuda":
            torch.cuda.synchronize()
        return self.nv.nvmlDeviceGetTotalEnergyConsumption(self.h) / 1000.0 if self.ok else time.perf_counter()

def time_energy(fn, meter, target_seconds, repeats, warmup=10):
    for _ in range(warmup):
        fn()
    meter.read()
    t0 = time.perf_counter()
    for _ in range(20):
        fn()
    meter.read()
    t_call = (time.perf_counter() - t0) / 20
    reps = max(30, int(target_seconds / max(t_call, 1e-6)))
    vals = []
    for _ in range(repeats):
        e0 = meter.read()
        for _ in range(reps):
            fn()
        e1 = meter.read()
        vals.append((e1 - e0) / reps)
    return float(np.median(vals))

class MACCostModel:
    # Analytic multiply-accumulates (includes the attention matmuls, which fvcore skips for fused attention).
    def __init__(self, geom, num_classes=NUM_CLASSES):
        self.D, self.Np, self.C = geom["dim"], geom["num_patches"], num_classes
    def embed(self): return self.Np * (16 * 16 * 3) * self.D
    def block(self, N): return 12 * N * self.D * self.D + 2 * N * N * self.D
    def head(self, arch):
        if arch["arch"] == "linear": return self.D * self.C
        h = arch["hidden"]; return self.D * h + h * self.C
    def merge(self, N, r):
        Np = N - 1; a = (Np + 1) // 2; b = Np // 2
        return a * b * self.D
    def final(self): return self.D * self.C

class TokenCostModel:
    # "token-layers": number of tokens pushed through transformer blocks (the transformer analogue of token cost)
    def embed(self): return 0.0
    def block(self, N): return float(N)
    def head(self, arch): return 0.0
    def merge(self, N, r): return 0.0
    def final(self): return 0.0

class MeasuredCostModel:
    def __init__(self, table):
        self.t = table
        self.unit = table.get("unit", "J")
        self.bk = sorted(int(k) for k in table["block"])
    def embed(self): return self.t["embed"]
    def block(self, N):
        return float(np.interp(N, self.bk, [self.t["block"][str(k)] for k in self.bk]))
    def head(self, arch): return self.t["head"][json.dumps(arch, sort_keys=True)]
    def merge(self, N, r):
        key = f"{N}_{r}"
        if key in self.t["merge"]:
            return self.t["merge"][key]
        cands = [(abs(int(k.split("_")[0]) - N), v) for k, v in self.t["merge"].items() if float(k.split("_")[1]) == r]
        return min(cands)[1] if cands else 0.0
    def final(self): return self.t["final"]

def path_costs(cfg, cm, geom):
    # cost (in the cost model's unit) of a sample that leaves at each exit, or runs to the end ('final').
    cfg = norm_cfg(cfg)
    exits = set(cfg["exit_layers"])
    N = 1 + geom["num_patches"]
    cost, costs = cm.embed(), {}
    for layer in range(1, geom["blocks"] + 1):
        cost += cm.block(N)
        if layer in exits:
            cost += cm.head(cfg["head_arch"][layer])
            costs[layer] = cost
            r = cfg["merge_ratio"].get(layer, 0.0)
            if r > 0:
                cost += cm.merge(N, r)
                N = 1 + merged_patch_count(N - 1, r)
    costs["final"] = cost + cm.final()
    return costs


In [ ]:
# ---------------- calibration, exit decisions, Pareto front, knee, operating points ----------------
def fit_temperature_grid(logits, labels):
    # robust: grid search over T minimising NLL (replaces the LBFGS version that collapsed to its lower clamp)
    grid = torch.logspace(-1.3, 1.0, 200)
    nll = torch.stack([F.cross_entropy(logits / t, labels) for t in grid])
    return float(grid[nll.argmin()].item())

def fit_all_temps(logits, labels):
    return {k: fit_temperature_grid(v, labels) for k, v in logits.items()}

def expected_calibration_error(conf, correct, n_bins=15):
    conf, correct = np.asarray(conf), np.asarray(correct).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def normalized_entropy(logits, T):
    p = F.softmax(logits / T, dim=-1)
    return -(p * torch.log(p.clamp_min(1e-12))).sum(-1) / math.log(logits.shape[1])

def sweep_points(logits, labels, temps, exit_layers, costs_by_model, taus, n_blocks, details=False):
    # Applies "exit at the first head whose calibrated normalised entropy < tau" OFFLINE to cached logits.
    # Identical to what forward_single_early_exit does per sample, but vectorised -> a whole tau sweep is cheap.
    exit_layers = sorted(exit_layers)
    N = labels.shape[0]
    ents = {l: normalized_entropy(logits[l], temps.get(l, 1.0)) for l in exit_layers}
    conf_exit = {l: F.softmax(logits[l] / temps.get(l, 1.0), -1).max(-1).values for l in exit_layers}
    pred_exit = {l: logits[l].argmax(-1) for l in exit_layers}
    pred_final = logits["final"].argmax(-1)
    conf_final = F.softmax(logits["final"] / temps.get("final", 1.0), -1).max(-1).values
    out = []
    for tau in taus:
        pred, conf = pred_final.clone(), conf_final.clone()
        used = torch.full((N,), n_blocks, dtype=torch.long)
        active = torch.ones(N, dtype=torch.bool)
        for l in exit_layers:
            m = active & (ents[l] < tau)
            pred[m], conf[m], used[m] = pred_exit[l][m], conf_exit[l][m], l
            active &= ~m
        frac = {str(l): float((used == l).float().mean()) for l in exit_layers}
        frac["final"] = float((used == n_blocks).float().mean())
        rec = {"tau": float(tau), "accuracy": float((pred == labels).float().mean()),
               "exit_fraction": frac, "mean_exit_layer": float(used.float().mean())}
        for name, cd in costs_by_model.items():
            per = torch.empty(N, dtype=torch.float64)
            per[used == n_blocks] = cd["final"]
            for l in exit_layers:
                per[used == l] = cd[l]
            rec[f"cost_{name}"] = float(per.mean())
        if details:
            rec.update(pred=pred.numpy(), conf=conf.numpy(), used=used.numpy())
        out.append(rec)
    return out

def pareto_front(points, cost_key):
    # maximise accuracy, minimise cost. Returned in ascending cost (and strictly ascending accuracy).
    front, best = [], -1.0
    for p in sorted(points, key=lambda p: (p[cost_key], -p["accuracy"])):
        if p["accuracy"] > best + 1e-12:
            front.append(p); best = p["accuracy"]
    return front

def knee_point(front, cost_key):
    # point of maximum distance above the chord of the normalised (cost, accuracy) curve
    if len(front) < 3:
        return front[0]
    c = np.array([p[cost_key] for p in front]); a = np.array([p["accuracy"] for p in front])
    if c.max() == c.min() or a.max() == a.min():
        return front[0]
    score = (a - a.min()) / (a.max() - a.min()) - (c - c.min()) / (c.max() - c.min())
    return front[int(score.argmax())]

def named_operating_points(points, cost_key, base_acc, eps):
    # conservative: cheapest point within 0.25*eps of baseline | knee: knee of the constraint-feasible front
    # aggressive: cheapest point within 2*eps (deliberately beyond the tolerance, to show what extra savings cost)
    feas = [p for p in points if p["accuracy"] >= base_acc - eps] or [max(points, key=lambda p: p["accuracy"])]
    wide = [p for p in points if p["accuracy"] >= base_acc - 2 * eps] or feas
    cons_pool = [p for p in points if p["accuracy"] >= base_acc - 0.25 * eps] or [max(points, key=lambda p: p["accuracy"])]
    return {"conservative": min(cons_pool, key=lambda p: p[cost_key]),
            "knee": knee_point(pareto_front(feas, cost_key), cost_key),
            "aggressive": pareto_front(wide, cost_key)[0]}

def marginal_table(points, cost_key):
    # walking from the most expensive front point to the cheapest: cost saved per accuracy POINT given up
    front = pareto_front(points, cost_key)[::-1]
    rows = []
    for prev, cur in zip(front[:-1], front[1:]):
        d_cost = prev[cost_key] - cur[cost_key]
        d_acc_pts = (prev["accuracy"] - cur["accuracy"]) * 100
        rows.append({"from_acc": prev["accuracy"], "to_acc": cur["accuracy"], "cost_saved": d_cost,
                     "acc_points_lost": d_acc_pts, "saved_per_acc_point": d_cost / max(d_acc_pts, 1e-9),
                     "to_cost": cur[cost_key], "to_key": cur.get("key"), "to_tau": cur["tau"]})
    # explicit columns: a one-point front gives no rows, and the CSV must still be readable
    return pd.DataFrame(rows, columns=["from_acc", "to_acc", "cost_saved", "acc_points_lost", "saved_per_acc_point", "to_cost", "to_key", "to_tau"])

def macro_f1(y_true, y_pred, n_classes=NUM_CLASSES):
    f1s = []
    for c in range(n_classes):
        tp = np.sum((y_pred == c) & (y_true == c)); fp = np.sum((y_pred == c) & (y_true != c)); fn = np.sum((y_pred != c) & (y_true == c))
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s))

def paired_bootstrap_diff(correct_a, correct_b, n=2000, seed=0):
    d = np.asarray(correct_a, float) - np.asarray(correct_b, float)
    rng = np.random.RandomState(seed)
    means = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(n)])
    return float(d.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


## 3. Splits + frozen backbone

In [ ]:
# ---------------- data splits (all disjoint) + backbone ----------------
Xtr, Ytr, Xte, Yte = load_cifar_tensors()
train_data = GPUData(Xtr, Ytr)
labels_np = Ytr.numpy()
_used = set()
def _take(per_class, seed):
    idx = stratified_indices(labels_np, per_class, seed, _used)
    _used.update(idx)
    return idx
search_train_idx = _take(SEARCH_TRAIN_PER_CLASS, SEED)
search_val_idx   = _take(SEARCH_VAL_PER_CLASS, SEED + 1)
search_cal_idx   = _take(SEARCH_CAL_PER_CLASS, SEED + 2)
final_train_idx  = _take(FINAL_TRAIN_PER_CLASS, SEED + 3)
final_cal_idx    = _take(FINAL_CAL_PER_CLASS, SEED + 4)
final_val_idx    = _take(FINAL_VAL_PER_CLASS, SEED + 5)
print("search  train/val/cal:", len(search_train_idx), len(search_val_idx), len(search_cal_idx))
print("final   train/val/cal:", len(final_train_idx), len(final_val_idx), len(final_cal_idx))

backbone, geom = make_backbone()
print("geometry:", geom)


## 4. Measured-energy cost model
Micro-benchmarks every building block (block at each reachable token count, merge op, each head, embed, final) on GPU 0 with the NVML energy counter.

In [ ]:
# ---------------- measured-energy cost model (one-off micro-benchmarks, GPU 0 via NVML) ----------------
meter = EnergyMeter()

def profile_cost_model(backbone, geom, meter):
    D, Np0 = geom["dim"], geom["num_patches"]
    blk = backbone.blocks[0]
    table = {"unit": meter.unit, "block": {}, "merge": {}, "head": {}}
    states, frontier, merge_pairs = {1 + Np0}, {1 + Np0}, set()   # every token count reachable through <= MAX_EXITS merges
    for _ in range(MAX_EXITS):
        new = set()
        for N in frontier:
            for r in MERGE_RATIO_POOL:
                if r > 0:
                    merge_pairs.add((N, r)); new.add(1 + merged_patch_count(N - 1, r))
        states |= new; frontier = new
    reps = dict(meter=meter, target_seconds=PROFILE_TARGET_SECONDS, repeats=3)
    with torch.no_grad(), amp_ctx():
        for N in sorted(states):
            x = torch.randn(1, N, D, device=device)
            table["block"][str(N)] = time_energy(lambda: blk(x), **reps)
        for (N, r) in sorted(merge_pairs):
            x = torch.randn(1, N, D, device=device)
            table["merge"][f"{N}_{r}"] = time_energy(lambda: bipartite_soft_matching_merge(x, r), **reps)
        img = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
        table["embed"] = time_energy(lambda: embed_tokens(backbone, img), **reps)
        cls = torch.randn(1, D, device=device)
        for arch in HEAD_ARCH_POOL:
            head = OffRampHead(D, NUM_CLASSES, **arch).to(device).eval()
            def head_fn():
                lg = head(cls).float()
                p = F.softmax(lg, -1)
                (-(p * torch.log(p.clamp_min(1e-12))).sum()).item()     # includes the exit-decision sync
            table["head"][json.dumps(arch, sort_keys=True)] = time_energy(head_fn, **reps)
        tok = torch.randn(1, 1 + Np0, D, device=device)
        fh = nn.Linear(D, NUM_CLASSES).to(device)
        table["final"] = time_energy(lambda: fh(backbone.norm(tok)[:, 0, :]).float().argmax().item(), **reps)
    return table

cost_path = f"{RESULTS_DIR}/cost_model_energy.json"
if os.path.exists(cost_path):
    cost_table = json.load(open(cost_path)); print("loaded existing cost model")
else:
    cost_table = profile_cost_model(backbone, geom, meter)
    json.dump(cost_table, open(cost_path, "w"), indent=2)

u = 1000 if cost_table["unit"] == "J" else 1
print(f"unit={cost_table['unit']}  (values below in milli-{cost_table['unit']} per call)")
print("embed:", cost_table["embed"] * u, "| final:", cost_table["final"] * u)
for N, v in cost_table["block"].items():
    print(f"  block @ {N:>4} tokens: {v * u:.4f}")
for k, v in cost_table["merge"].items():
    print(f"  merge @ {k}: {v * u:.4f}")
for k, v in cost_table["head"].items():
    print(f"  head {k}: {v * u:.4f}")

cost_models = {"energy": MeasuredCostModel(cost_table), "macs": MACCostModel(geom), "tokens": TokenCostModel()}
def costs_for(cfg):
    return {name: path_costs(cfg, cm, geom) for name, cm in cost_models.items()}


## 5. Training utilities + Phase 0 sanity gate

In [ ]:
# ---------------- training / logits utilities ----------------
def make_scaler():
    on = USE_AMP and device == "cuda"
    try:
        return torch.amp.GradScaler("cuda", enabled=on)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=on)

def train_heads(model, data, idx, epochs, lr=LR, verbose=True):
    # backbone frozen (linear-probe style); heads trained with equal-weight joint CE (NACHOS Eq.7 style)
    params = list(model.final_head.parameters()) + [p for h in model.heads.values() for p in h.parameters()]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    steps = epochs * math.ceil(len(idx) / BATCH_SIZE)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    scaler = make_scaler()
    model.train(); model.backbone.eval()
    for ep in range(epochs):
        tot, n = 0.0, 0
        for x, y in data.batches(idx, BATCH_SIZE, shuffle=True, augment=True):
            with amp_ctx():
                logits = model(x)
            loss = F.cross_entropy(logits["final"], y)
            for l in model.exit_layers:
                loss = loss + F.cross_entropy(logits[l], y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * x.size(0); n += x.size(0)
        if verbose:
            print(f"    epoch {ep + 1}/{epochs} loss={tot / n:.4f}")
    model.eval()
    return model

def seed_for(cfg):
    torch.manual_seed(int(cfg_hash(cfg), 16) % (2 ** 31))

B0_MIN_VAL_ACC = 0.90   # a correctly normalised frozen ViT-Tiny linear probe reaches about 0.93 on this split size

train_log_path = f"{RESULTS_DIR}/training_log.json"
train_log = json.load(open(train_log_path)) if os.path.exists(train_log_path) else {}

B0_cfg = {"exit_layers": [], "head_arch": {}, "merge_ratio": {}}
_mid = EXIT_LAYER_POOL[len(EXIT_LAYER_POOL) // 2]
B1_cfg = {"exit_layers": [_mid], "head_arch": {_mid: {"arch": "linear"}}, "merge_ratio": {_mid: 0.0}}   # fixed mid-depth exit

def get_final_model(cfg):
    cfg = norm_cfg(cfg); h = cfg_hash(cfg)
    path = f"{CHECKPOINT_DIR}/model_{h}.pt"
    if os.path.exists(path):
        print("  resumed from checkpoint:", cfg_label(cfg))
        return load_ckpt(path, backbone)[0]
    print("  training final model:", cfg_label(cfg))
    seed_for(cfg)
    model = EarlyExitViT(backbone, NUM_CLASSES, **cfg).to(device)
    t0 = time.time()
    train_heads(model, train_data, final_train_idx, FINAL_EPOCHS)
    cal_logits, cal_labels = collect_logits(model, train_data, final_cal_idx)
    model.temps = fit_all_temps(cal_logits, cal_labels)
    save_ckpt(model, cfg, model.temps, path)
    train_log[h] = {"label": cfg_label(cfg), "seconds": round(time.time() - t0, 1), "train_samples": len(final_train_idx), "epochs": FINAL_EPOCHS}
    json.dump(train_log, open(train_log_path, "w"), indent=2)
    return model


In [ ]:
# ---------------- Phase 0: B0 on the FINAL budget + sanity gate ----------------
# If the frozen ViT-Tiny + linear head cannot reach a sensible accuracy here, STOP: depth comparisons are meaningless.
B0_model = get_final_model(B0_cfg)
_lg, _lb = collect_logits(B0_model, train_data, final_val_idx)
b0_val_acc = float((_lg["final"].argmax(-1) == _lb).float().mean())
print(f"\nB0 (final head only) validation accuracy: {b0_val_acc:.4f}   [{len(final_train_idx)} train samples, {FINAL_EPOCHS} epochs]")
if b0_val_acc < B0_MIN_VAL_ACC and not SMOKE_TEST:
    raise RuntimeError(f"B0 accuracy {b0_val_acc:.4f} is below {B0_MIN_VAL_ACC}. Expected about 0.93. Fix data/normalisation before spending GPU time.")


## 6. AutoML search (resumable)

In [ ]:
# ---------------- Phase 1: AutoML search (custom NSGA-II-style loop; resumable) ----------------
rng = np.random.RandomState(SEED)

def rand_head(): return dict(HEAD_ARCH_POOL[rng.randint(len(HEAD_ARCH_POOL))])
def rand_merge(): return float(MERGE_RATIO_POOL[rng.randint(len(MERGE_RATIO_POOL))])

def sample_config():
    n = rng.randint(1, MAX_EXITS + 1)
    ls = sorted(rng.choice(EXIT_LAYER_POOL, size=n, replace=False).tolist())
    return {"exit_layers": ls, "head_arch": {l: rand_head() for l in ls}, "merge_ratio": {l: rand_merge() for l in ls}}

def mutate_config(cfg):
    cfg = norm_cfg(cfg)
    move = rng.choice(["add_exit", "remove_exit", "change_head", "change_merge", "move_exit"])
    ls = cfg["exit_layers"]
    if move == "add_exit" and len(ls) < MAX_EXITS:
        free = [l for l in EXIT_LAYER_POOL if l not in ls]
        if free:
            l = int(free[rng.randint(len(free))])
            cfg["exit_layers"] = sorted(ls + [l]); cfg["head_arch"][l] = rand_head(); cfg["merge_ratio"][l] = rand_merge()
    elif move == "remove_exit" and len(ls) > 1:
        l = ls[rng.randint(len(ls))]
        cfg["exit_layers"].remove(l); cfg["head_arch"].pop(l); cfg["merge_ratio"].pop(l)
    elif move == "change_head":
        cfg["head_arch"][ls[rng.randint(len(ls))]] = rand_head()
    elif move == "change_merge":
        cfg["merge_ratio"][ls[rng.randint(len(ls))]] = rand_merge()
    elif move == "move_exit":
        l = ls[rng.randint(len(ls))]
        free = [x for x in EXIT_LAYER_POOL if x not in ls]
        if free:
            nl = int(free[rng.randint(len(free))])
            cfg["exit_layers"] = sorted([x for x in ls if x != l] + [nl])
            cfg["head_arch"][nl] = cfg["head_arch"].pop(l); cfg["merge_ratio"][nl] = cfg["merge_ratio"].pop(l)
    return norm_cfg(cfg)

def evaluate_config(cfg):
    cfg = norm_cfg(cfg); h = cfg_hash(cfg)
    seed_for(cfg)
    model = EarlyExitViT(backbone, NUM_CLASSES, **cfg).to(device)
    train_heads(model, train_data, search_train_idx, SEARCH_EPOCHS, verbose=False)
    cal_logits, cal_labels = collect_logits(model, train_data, search_cal_idx)
    temps = fit_all_temps(cal_logits, cal_labels)
    val_logits, val_labels = collect_logits(model, train_data, search_val_idx)
    pts = sweep_points(val_logits, val_labels, temps, cfg["exit_layers"], costs_for(cfg), TAU_GRID, geom["blocks"])
    for p in pts:
        p["config"] = cfg; p["key"] = h
    return pts

def elite_configs(points, base_acc):
    feas = [p for p in points if p["accuracy"] >= base_acc - EPS] or sorted(points, key=lambda p: -p["accuracy"])[:6]
    keep = {}
    for ck in ["cost_energy", "cost_macs"]:          # a config is elite if it is on EITHER front
        for p in pareto_front(feas, ck):
            keep[p["key"]] = p["config"]
    return list(keep.values())

# proxy baseline (final head only) at the SEARCH budget: the accuracy floor for the constraint
search_base_acc = evaluate_config(B0_cfg)[0]["accuracy"]
print(f"proxy baseline accuracy (search budget): {search_base_acc:.4f}")

state_path = f"{RESULTS_DIR}/search_state.json"
points, evaluated, start_gen, population = [], {}, 0, None
if os.path.exists(state_path):
    st = json.load(open(state_path))
    points, start_gen = st["points"], st["gen_done"]
    for p in points:
        p["config"] = norm_cfg(p["config"]); evaluated.setdefault(p["key"], []).append(p)
    population = [norm_cfg(c) for c in st["population"]]
    print(f"resuming search from generation {start_gen}, {len(evaluated)} configs already evaluated")
if population is None:
    population = [sample_config() for _ in range(SEARCH_POP)]

for gen in range(start_gen, SEARCH_GENS):
    print(f"\n=== generation {gen + 1}/{SEARCH_GENS}: {len(population)} candidates ===")
    t0 = time.time()
    for cfg in population:
        h = cfg_hash(cfg)
        if h in evaluated:
            continue
        pts = evaluate_config(cfg)
        evaluated[h] = pts; points.extend(pts)
        best = max(p["accuracy"] for p in pts)
        print(f"   {cfg_label(cfg):<46} best acc={best:.4f}")
    elites = elite_configs(points, search_base_acc)
    seen_now = {cfg_hash(c) for c in population}
    nxt = []
    for _ in range(300):
        if len(nxt) >= max(1, SEARCH_POP - 2): break
        c = mutate_config(elites[rng.randint(len(elites))])
        if cfg_hash(c) not in evaluated and cfg_hash(c) not in {cfg_hash(x) for x in nxt}:
            nxt.append(c)
    for _ in range(300):
        if len(nxt) >= SEARCH_POP: break
        c = sample_config()
        if cfg_hash(c) not in evaluated and cfg_hash(c) not in {cfg_hash(x) for x in nxt}:
            nxt.append(c)
    population = nxt
    json.dump({"gen_done": gen + 1, "population": population, "points": points}, open(state_path, "w"))
    print(f"   generation took {time.time() - t0:.0f}s | configs evaluated: {len(evaluated)} | elites: {len(elites)}")
    if not population:
        print("search space exhausted"); break

flat = pd.DataFrame([{"key": p["key"], "label": cfg_label(p["config"]), "tau": p["tau"], "accuracy": p["accuracy"],
                      "mean_exit_layer": p["mean_exit_layer"], "cost_energy": p["cost_energy"], "cost_macs": p["cost_macs"],
                      "cost_tokens": p["cost_tokens"]} for p in points])
flat.to_csv(f"{RESULTS_DIR}/search_points.csv", index=False)
print("saved search_points.csv:", len(flat), "rows")


## 7. Candidate selection

In [ ]:
# ---------------- candidate selection from the search archive ----------------
named_E = named_operating_points(points, "cost_energy", search_base_acc, EPS)   # cost = MEASURED energy
named_M = named_operating_points(points, "cost_macs", search_base_acc, EPS)     # cost = analytic MACs (NACHOS-style cost)
for title, named in [("ENERGY-driven", named_E), ("MAC-driven", named_M)]:
    print(f"\n{title} search picks (proxy numbers, tau selected on search-val):")
    for k, p in named.items():
        print(f"  {k:<13} {cfg_label(p['config']):<46} tau={p['tau']:<5} acc={p['accuracy']:.4f}")
cand = {"energy": {p["key"]: p["config"] for p in named_E.values()},
        "mac": {p["key"]: p["config"] for p in named_M.values()}}


## 8. Final models + calibration

In [ ]:
# ---------------- Phase 2: train the final models (resumable) + calibrate ----------------
all_cfgs = {cfg_hash(B0_cfg): norm_cfg(B0_cfg), cfg_hash(B1_cfg): norm_cfg(B1_cfg)}
for pipe in cand:
    all_cfgs.update(cand[pipe])
print(len(all_cfgs), "distinct final models:", [cfg_label(c) for c in all_cfgs.values()])

models = {}
for h, cfg in all_cfgs.items():
    models[h] = B0_model if h == cfg_hash(B0_cfg) else get_final_model(cfg)


## 9. τ sweep on validation -> operating points + marginal-efficiency tables

In [ ]:
# ---------------- tau sweep on the held-out FINAL validation split -> operating points ----------------
final_points, calib_report = {}, {}
for h, cfg in all_cfgs.items():
    m = models[h]
    v_logits, v_labels = collect_logits(m, train_data, final_val_idx)
    pts = sweep_points(v_logits, v_labels, m.temps, cfg["exit_layers"], costs_for(cfg), TAU_GRID, geom["blocks"])
    for p in pts:
        p["config"] = cfg; p["key"] = h
    final_points[h] = pts
    rep = {"label": cfg_label(cfg), "temps": {str(k): v for k, v in m.temps.items()}, "ece_before": {}, "ece_after": {}}
    for k, lg in v_logits.items():
        correct = (lg.argmax(-1) == v_labels).numpy()
        rep["ece_before"][str(k)] = expected_calibration_error(F.softmax(lg, -1).max(-1).values.numpy(), correct)
        rep["ece_after"][str(k)] = expected_calibration_error(F.softmax(lg / m.temps.get(k, 1.0), -1).max(-1).values.numpy(), correct)
    calib_report[h] = rep
json.dump(calib_report, open(f"{RESULTS_DIR}/calibration_results.json", "w"), indent=2)

base_val_acc = final_points[cfg_hash(B0_cfg)][0]["accuracy"]
print(f"\nB0 validation accuracy (reference for the no-regression rule): {base_val_acc:.4f}")

def tau_by_constraint(pts, cost_key):
    feas = [p for p in pts if p["accuracy"] >= base_val_acc - EPS]
    return min(feas, key=lambda p: p[cost_key]) if feas else min(pts, key=lambda p: p["tau"])

ops = {}
def add_op(name, pipeline, p):
    ops[name] = {"pipeline": pipeline, "config": p["config"], "hash": p["key"], "tau": p["tau"],
                 "val_accuracy": p["accuracy"], "exit_fraction": p["exit_fraction"],
                 **{k: v for k, v in p.items() if k.startswith("cost_")}}

add_op("B0", "baseline", final_points[cfg_hash(B0_cfg)][0])
add_op("B1", "fixed-exit", tau_by_constraint(final_points[cfg_hash(B1_cfg)], "cost_energy"))
for pipe, cost_key, prefix in [("energy", "cost_energy", "E"), ("mac", "cost_macs", "M")]:
    pool = [p for h in cand[pipe] for p in final_points[h]]
    named = named_operating_points(pool, cost_key, base_val_acc, EPS)
    for k, short in [("conservative", "cons"), ("knee", "knee"), ("aggressive", "aggr")]:
        add_op(f"{prefix}_{short}", pipe, named[k])
    front = pareto_front(pool, cost_key)
    pd.DataFrame([{"key": p["key"], "label": cfg_label(p["config"]), "tau": p["tau"], "accuracy": p["accuracy"],
                   **{k: v for k, v in p.items() if k.startswith("cost_")}} for p in front]).to_csv(f"{RESULTS_DIR}/front_{pipe}_val.csv", index=False)
    marginal_table(pool, cost_key).to_csv(f"{RESULTS_DIR}/marginal_{pipe}_val.csv", index=False)

json.dump(ops, open(f"{RESULTS_DIR}/operating_points.json", "w"), indent=2)
pd.DataFrame([{"key": p["key"], "label": cfg_label(p["config"]), "tau": p["tau"], "accuracy": p["accuracy"],
               "mean_exit_layer": p["mean_exit_layer"], **{k: v for k, v in p.items() if k.startswith("cost_")}}
              for h in final_points for p in final_points[h]]).to_csv(f"{RESULTS_DIR}/val_sweep_points.csv", index=False)

summary = pd.DataFrame([{"name": n, "config": cfg_label(o["config"]), "tau": o["tau"], "val_acc": round(o["val_accuracy"], 4),
                         "energy_pred_mJ": round(o["cost_energy"] * (1000 if cost_table["unit"] == "J" else 1), 4),
                         "macs_M": round(o["cost_macs"] / 1e6, 1)} for n, o in ops.items()])
print(summary.to_string(index=False))
print("\nmarginal-efficiency table (energy-driven front, validation):")
print(pd.read_csv(f"{RESULTS_DIR}/marginal_energy_val.csv").round(5).to_string(index=False))


## 10. Package outputs

In [ ]:
# ---------------- zip everything for download ----------------
zip_path = os.path.join(WORKING_DIR, "early_exit_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [CHECKPOINT_DIR, RESULTS_DIR]:
        for f in sorted(os.listdir(folder)):
            z.write(os.path.join(folder, f), arcname=os.path.join(os.path.basename(folder), f))
print("zip written:", zip_path, f"({os.path.getsize(zip_path) / 1024:.0f} KB)")
for folder in [CHECKPOINT_DIR, RESULTS_DIR]:
    print(folder)
    for f in sorted(os.listdir(folder)):
        print("   ", f)
print("\nNext: run evaluate.ipynb (same session, or attach this notebook's output as input data).")
print("Kaggle: use Save Version > Save & Run All, then fetch with: kaggle kernels output <user>/<notebook-slug> -p ./outputs")
